# Task 1: Analysis of Drug–Target Datasets

## Objective: 
Explore score distributions and observation of the drug-target pairs. Explain the main features and key differences between DAVIS and KIBA.

## Notebook structure
This ipynb will be split into 5 parts:
1. DAVIS dataset
2. KIBA dataset
3. DAVIS vs KIBA comparison
4. Comparison with CSV files for both DAVIS and KIBA

For parts 1 to 3, the data used is from the [data github repo](https://github.com/hkmztrk/DeepDTA/tree/master/data)
The CSV files for part 4 were taken from [the kaggle dataset](https://www.kaggle.com/datasets/rajaryan2315/davis-and-kiba-datasets)

## Import and file locations

In [14]:
import os, json, pickle, urllib.request, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap
from scipy.stats import spearmanr
from scipy.cluster.hierarchy import linkage, leaves_list
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import rdFingerprintGenerator
RDLogger.DisableLog('rdApp.*'); warnings.filterwarnings('ignore')

SEED = 6767
np.random.seed(SEED)
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.width', 160, 'display.max_colwidth', 70)

DAVIS_DIR = '../Data/davis/'
KIBA_DIR = '../Data/kiba/'

LIMIT_KD_NM = 10_000.0
LIMIT_PKD   = 5

## Helper functions

Deduplication helpers

In [15]:
def canon(smi):
    """
    put the SMILES into canonical form, but if cant find then return the original
    """
    m = Chem.MolFromSmiles(smi)
    return Chem.MolToSmiles(m) if m is not None else smi

def counts(df, lig='lig_key', tgt='tgt_key'):
    """
    num unique ligands, targets, and pairs
    """
    return dict(ligands = df[lig].nunique(), 
                targets = df[tgt].nunique(), 
                observed_pairs = len(df))

def clean_and_dedup(long_df, score_col, positive_required, transform=None):
    """
    long_df cols should be 
        ligand_cid,
        target,
        smiles,
        sequence, 
        <score_col>

    out: out - cleaned dataset where one row = one ligand-target pair
    table: table - summary of how many ligands, targets, and pairs remain after each cleaning step
    """
    
    log = []
    df = long_df.copy()
    df['lig_key'] = df.ligand_cid; df['tgt_key'] = df.target
    log.append(('0. raw records (matrix cells including missing)', dict(ligands=df.lig_key.nunique(), targets=df.tgt_key.nunique(), observed_pairs=len(df))))
    df = df[df[score_col].notna()]
    
    log.append(('1. observed values (not NaN)', counts(df)))
    fin = np.isfinite(df[score_col]); n_nonfinite = int((~fin).sum()); df = df[fin]
    
    log.append((f'2. finite scores (dropped {n_nonfinite})', counts(df)))
    
    if positive_required:
        pos = df[score_col] > 0; n_nonpos = int((~pos).sum()); df = df[pos]
        log.append((f'3. Kd > 0 (dropped {n_nonpos})', counts(df)))
    df = df.assign(score=transform(df[score_col]) if transform else df[score_col])
    
    # --- de-duplication keys ---
    # apply canon to SMILES then use that as the key
    canonMap = {x: canon(x) for x in df.smiles.unique()}
    df['lig_key'] = df.smiles.map(canonMap)
    df['tgt_key'] = df.sequence
    log.append(('4. keyed by canonical SMILES / sequence (before averaging)', counts(df)))
    g = df.groupby(['lig_key', 'tgt_key'])
    out = g.agg(score=('score', 'mean'), n_repeats=('score', 'size'), score_min=('score', 'min'), score_max=('score', 'max'),
                ligand_cid=('ligand_cid', 'first'), target=('target', 'first')).reset_index()
    out['score_spread'] = out.score_max - out.score_min
    log.append(('5. deduplicated, repeats averaged', counts(out)))
    table = pd.DataFrame({k: v for k, v in log}).T
    return out, table


## Part 1: DAVIS dataset

### 1.1 Load dataset variables

In [16]:
ligands = json.load(open(os.path.join(DAVIS_DIR, 'ligands_can.txt')))
proteins = json.load(open(os.path.join(DAVIS_DIR, 'proteins.txt')))
with open(f'{DAVIS_DIR}/Y', 'rb') as temp:
    davis_Y = pickle.load(temp, encoding='latin1')

print("pickled davis_Y shape:", davis_Y.shape, davis_Y.dtype)
print("ligands:", len(ligands), "proteins:", len(proteins))
lig_ids = list(ligands)
tgt_names = list(proteins)

raw_df = pd.DataFrame(davis_Y, index=lig_ids, columns=tgt_names)
long_raw = raw_df.rename_axis(index='ligand_cid', columns='target').reset_index().melt(id_vars='ligand_cid', var_name='target', value_name='Kd_nM')
long_raw['smiles'] = long_raw.ligand_cid.map(ligands)
long_raw['sequence'] = long_raw.target.map(proteins)
short = lambda d: d.assign(sequence=d.sequence.str[:30] + '...', smiles=d.smiles.str[:40] + '...')

print("---\nfirst 5 rows of long_raw:")
display(short(long_raw).head(5))
print("top left corner of Kd matrix:")
display(raw_df.iloc[:5, :6])

pickled davis_Y shape: (68, 442) float64
ligands: 68 proteins: 442
---
first 5 rows of long_raw:


,ligand_cid,target,Kd_nM,smiles,sequence
0,11314340,AAK1,43.0,CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OCC(CC4...,MKKFFDSRREQGGSGLGSGSSGGGGSTSGL...
1,24889392,AAK1,10000.0,CC(C)(C)C1=CC(=NO1)NC(=O)NC2=CC=C(C=C2)C...,MKKFFDSRREQGGSGLGSGSSGGGGSTSGL...
2,11409972,AAK1,10000.0,CCN1CCN(CC1)CC2=C(C=C(C=C2)NC(=O)NC3=CC=...,MKKFFDSRREQGGSGLGSGSSGGGGSTSGL...
3,11338033,AAK1,10000.0,C1CNCCC1NC(=O)C2=C(C=NN2)NC(=O)C3=C(C=CC...,MKKFFDSRREQGGSGLGSGSSGGGGSTSGL...
4,10184653,AAK1,10000.0,CN(C)CC=CC(=O)NC1=C(C=C2C(=C1)C(=NC=N2)N...,MKKFFDSRREQGGSGLGSGSSGGGGSTSGL...


top left corner of Kd matrix:


,AAK1,ABL1(E255K),ABL1(F317I),ABL1(F317I)p,ABL1(F317L),ABL1(F317L)p
11314340,43.0,10000.0,10000.0,10000.0,10000.00,10000.0
24889392,10000.0,10000.0,10000.0,10000.0,10000.00,10000.0
11409972,10000.0,75.0,1.9,13.0,0.77,13.0
11338033,10000.0,10000.0,10000.0,10000.0,10000.00,10000.0
10184653,10000.0,420.0,2900.0,750.0,580.00,230.0


### 1.2 Dataset cleaning and deduplication

#### Cleaning steps
1. count observed = num of cells in matrix not NaN
2. remove and NaN or infinite values
3. remove negative Kd
4. convert Kd to pKd
5. deduplicate by using the canonical SMILES and targets
* rows with identical (ligand, target) pairs are averaged in pKd space

the functions are defined earlier under Helper Functions

In [17]:
davis_clean, davis_log = clean_and_dedup(long_raw, 'Kd_nM', positive_required = True, transform = lambda kd: 9 - np.log10(kd))
display(davis_log)
print('(ligand, target) cells that were averaged from repeats:', int((davis_clean.n_repeats > 1).sum()))

,ligands,targets,observed_pairs
0. raw records (matrix cells including missing),68,442,30056
1. observed values (not NaN),68,442,30056
2. finite scores (dropped 0),68,442,30056
3. Kd > 0 (dropped 0),68,442,30056
4. keyed by canonical SMILES / sequence (before averaging),68,379,30056
"5. deduplicated, repeats averaged",68,379,25772


(ligand, target) cells that were averaged from repeats: 1224


Compare original values with deduplicated values

In [23]:
orig, dedup = davis_log.loc['1. observed values (not NaN)'], davis_log.iloc[-1]
cells_all = lambda r: r.ligands * r.targets

summary = pd.DataFrame({'original': orig, 'deduplicated': dedup}).astype(float)
display(summary.style.format('{:.0f}'))

,original,deduplicated
ligands,68,68
targets,442,379
observed_pairs,30056,25772


### 1.3 Rebuild deduplicated matrix

In [26]:
def to_matrix(clean_data, row = "lig_key", col = "tgt_key", row_label = "ligand_cid", col_label = "target"):
    # Create a matrix of scores, with ligands as rows and targets as columns
    matrix = clean_data.pivot(
        index = row,
        columns = col,
        values = "score"
    )

    ligand_labels = clean_data.drop_duplicates(row).set_index(row)[row_label]
    target_labels = clean_data.drop_duplicates(col).set_index(col)[col_label]

    # Replace the ID with labels
    matrix.index = matrix.index.map(ligand_labels)
    matrix.columns = matrix.columns.map(target_labels)

    return matrix


def sparsity(matrix):
    # num missing / total
    missing_values = matrix.isna().sum().sum()
    total_values = matrix.size
    return missing_values / total_values


# Create the pKd matrix (ligands × targets)
pkd_matrix = to_matrix(davis_clean)

# Display basic information about the matrix
minimum_pkd = np.nanmin(pkd_matrix.values)
maximum_pkd = np.nanmax(pkd_matrix.values)

print("pKd matrix shape:", pkd_matrix.shape)
print("Sparsity:", round(sparsity(pkd_matrix), 3))
print("Minimum pKd:", round(minimum_pkd, 3))
print("Maximum pKd:", round(maximum_pkd, 3))

# Save the matrix to a CSV file
os.makedirs("data/processed", exist_ok=True)

pkd_matrix.to_csv("data/processed/davis_pkd_clean.csv")

# Extract all individual pKd scores
scores = davis_clean["score"].values

pKd matrix shape: (68, 379)
Sparsity: 0.0
Minimum pKd: 5.0
Maximum pKd: 10.62


### 1.4 Visualization for DAVIS
#### Ligand and target heatmap

use hierarchical clustering to reorder the rows and columns to make block structure clear
any missing values are greyed out